In [4]:
import json
import numpy as np

def _generate_curv_table_empirical(file_lookup, datasets, save_path, index, label):
    def format_value(val, unit="", highlight=None):
        if val == "-":
            return "-"
        v = f"{val:.2f}\\%" if unit == "%" else f"{val:.4f}"
        if highlight == "bold":
            return r"\textbf{" + v + "}"
        elif highlight == "underline":
            return r"\underline{" + v + "}"
        return v

    alphas = [0, 1]
    gammas = [0, 1, 100]
    dtopos = [0, 1, 2]
    all_rows = []

    for alpha in alphas:
        for gamma in gammas:
            if (gamma == 0 and alpha == 0) or (gamma == 100 and alpha == 0):
                continue
            for d in dtopos:
                if (alpha == 1 and gamma == 0 and d in {1, 2}):
                    continue
                meta = (f"{alpha}", f"{gamma}", f"{d}")
                mse_vals, smape_vals = [], []

                for ds in datasets:
                    path = file_lookup.get(ds, {}).get((alpha, gamma, d), None)
                    if not path:
                        mse_vals.append("-")
                        smape_vals.append("-")
                        continue
                    try:
                        with open(path, "r") as f:
                            data = json.load(f)
                        mse = data["errors"]["MSE"][index]
                        smape = data["errors"]["SMAPE_percent"][index]
                        mse_vals.append(mse)
                        smape_vals.append(smape)
                    except Exception as e:
                        print(f"Error reading {path}: {e}")
                        mse_vals.append("-")
                        smape_vals.append("-")

                all_rows.append((meta, mse_vals, smape_vals))

    # Highlight minima
    n_datasets = len(datasets)
    mse_col, smape_col = [[] for _ in range(n_datasets)], [[] for _ in range(n_datasets)]
    for rid, (_, mse_vals, smape_vals) in enumerate(all_rows):
        for i in range(n_datasets):
            if mse_vals[i] != "-":
                mse_col[i].append((rid, mse_vals[i]))
            if smape_vals[i] != "-":
                smape_col[i].append((rid, smape_vals[i]))

    mse_highlight, smape_highlight = {}, {}
    for i in range(n_datasets):
        if mse_col[i]:
            sorted_m = sorted(mse_col[i], key=lambda x: x[1])
            mse_highlight[(sorted_m[0][0], i)] = "bold"
            if len(sorted_m) > 1:
                mse_highlight[(sorted_m[1][0], i)] = "underline"
        if smape_col[i]:
            sorted_s = sorted(smape_col[i], key=lambda x: x[1])
            smape_highlight[(sorted_s[0][0], i)] = "bold"
            if len(sorted_s) > 1:
                smape_highlight[(sorted_s[1][0], i)] = "underline"

    # Render LaTeX
    lines = []
    lines.append(r"\begin{table}[ht]")
    lines.append(r"\scriptsize")
    lines.append(r"    \centering")
    lines.append(f"    \\begin{{tabular}}{{lll|{'cc|' * n_datasets}}}")
    lines.append(r"        \toprule")
    lines.append("        $\\alpha$ & $\\gamma$ & $d_{\\text{topo}}$" +
                 "".join([f" & \\multicolumn{{2}}{{c|}}{{{ds}}}" for ds in datasets]) + r" \\")
    lines.append("        & & & " + " & ".join(["MSE & SMAPE"] * n_datasets) + r" \\")
    lines.append(r"        \midrule")

    for rid, (meta, mse_vals, smape_vals) in enumerate(all_rows):
        alpha, gamma, dtopo = meta
        dtopo_display = "-" if gamma == "0" else dtopo
        row = [alpha, gamma, dtopo_display]
        for i in range(n_datasets):
            mse_fmt = format_value(mse_vals[i], highlight=mse_highlight.get((rid, i)))
            smape_fmt = format_value(smape_vals[i], unit="%", highlight=smape_highlight.get((rid, i)))
            row.extend([mse_fmt, smape_fmt])
        lines.append("        " + " & ".join(row) + r" \\")

    lines.append(r"        \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(rf"    \caption{{Curvature metrics from index {index} across models with varying $\alpha$, $\gamma$, $d_{{\text{{topo}}}}$.}}")
    lines.append(rf"    \label{{tab:{label}}}")
    lines.append(r"\end{table}")

    with open(save_path, "w") as f:
        f.write("\n".join(lines))
    print(f"LaTeX table written to: {save_path}")

def generate_curv_table_from_json_lookup_alpha_empirical(file_lookup, datasets, save_path):
    _generate_curv_table_empirical(file_lookup, datasets, save_path, index=0, label="curv_alpha_empirical")

def generate_curv_table_from_json_lookup_alpha_empirical_norm(file_lookup, datasets, save_path):
    _generate_curv_table_empirical(file_lookup, datasets, save_path, index=1, label="curv_alpha_empirical_norm")


In [5]:
base_path_flower = "notebooks_euclid_ae/results/s1_synthetic/"
base_path_scrunchy = "notebooks_euclid_ae/results/scrunchy_dim_n/"
base_path_sphere_low = "notebooks_euclid_ae/results/s2_synthetic/"
base_path_sphere_high = "notebooks_euclid_ae/results/sphere/"
base_path_torus_low= "notebooks_euclid_ae/results/t2_synthetic/"
base_path_torus_high= "notebooks_euclid_ae/results/torus/"
file_lookup = {
    "Curve low dim": {
        (1, 0, 0): base_path_flower + "results_exp01_ae_flower_curve_w100_b64/curvature_errors_stats.json",
        (1, 1, 0): base_path_flower + "results_exp02_ae_flower_curve_w100_b64/curvature_errors_stats.json",
        (1, 100, 0): base_path_flower + "results_exp03_ae_flower_curve_w100_b64/curvature_errors_stats.json",
        (0, 1, 0): base_path_flower + "results_exp04_ae_flower_curve_w100_b64/curvature_errors_stats.json",
        (1, 1, 1): base_path_flower + "results_exp05_ae_flower_curve_w100_b64/curvature_errors_stats.json",
        (1, 100, 1): base_path_flower + "results_exp06_ae_flower_curve_w100_b64/curvature_errors_stats.json",
        (0, 1, 1): base_path_flower + "results_exp07_ae_flower_curve_w100_b64/curvature_errors_stats.json",
        (1, 1, 2): base_path_flower + "results_exp08_ae_flower_curve_w100_b64/curvature_errors_stats.json",
        (1, 100, 2): base_path_flower + "results_exp09_ae_flower_curve_w100_b64/curvature_errors_stats.json",
        (0, 1, 2): base_path_flower + "results_exp10_ae_flower_curve_w100_b64/curvature_errors_stats.json",
    },
    "Curve high dim": {
        (1, 0, 0): base_path_scrunchy + "results_exp01_ae_scrunchy_w100_b64/curvature_errors_stats.json",
        (1, 1, 0): base_path_scrunchy + "results_exp02_ae_scrunchy_w100_b64/curvature_errors_stats.json",
        (1, 100, 0): base_path_scrunchy + "results_exp03_ae_scrunchy_w100_b64/curvature_errors_stats.json",
        (0, 1, 0): base_path_scrunchy + "results_exp04_ae_scrunchy_w100_b64/curvature_errors_stats.json",
        (1, 1, 1): base_path_scrunchy + "results_exp05_ae_scrunchy_w100_b64/curvature_errors_stats.json",
        (1, 100, 1): base_path_scrunchy + "results_exp06_ae_scrunchy_w100_b64/curvature_errors_stats.json",
        (0, 1, 1): base_path_scrunchy + "results_exp07_ae_scrunchy_w100_b64/curvature_errors_stats.json",
        (1, 1, 2): base_path_scrunchy + "results_exp08_ae_scrunchy_w100_b64/curvature_errors_stats.json",
        (1, 100, 2): base_path_scrunchy + "results_exp09_ae_scrunchy_w100_b64/curvature_errors_stats.json",
        (0, 1, 2): base_path_scrunchy + "results_exp10_ae_scrunchy_w100_b64/curvature_errors_stats.json",
    },
    "Sphere low dim": {
        (1, 0, 0): base_path_sphere_low + "results_exp02_ae_s2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 1, 0): base_path_sphere_low + "results_exp03_ae_s2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 100, 0): base_path_sphere_low + "results_exp04_ae_s2_synthetic_w100_b64/curvature_errors_stats.json",
        (0, 1, 0): base_path_sphere_low + "results_exp05_ae_s2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 1, 1): base_path_sphere_low + "results_exp06_ae_s2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 100, 1): base_path_sphere_low + "results_exp07_ae_s2_synthetic_w100_b64/curvature_errors_stats.json",
        (0, 1, 1): base_path_sphere_low + "results_exp08_ae_s2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 1, 2): base_path_sphere_low + "results_exp09_ae_s2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 100, 2): base_path_sphere_low + "results_exp10_ae_s2_synthetic_w100_b64/curvature_errors_stats.json",
        (0, 1, 2): base_path_sphere_low + "results_exp11_ae_s2_synthetic_w100_b64/curvature_errors_stats.json",
    },
    "Sphere high dim": {
        (1, 0, 0): base_path_sphere_high + "results_exp02_sphere_high_dim_new_version/curvature_errors_stats.json",
        (1, 1, 0): base_path_sphere_high + "results_exp03_sphere_high_dim_new_version/curvature_errors_stats.json",
        (1, 100, 0): base_path_sphere_high + "results_exp04_sphere_high_dim_new_version/curvature_errors_stats.json",
        (0, 1, 0): base_path_sphere_high + "results_exp05_sphere_high_dim_new_version/curvature_errors_stats.json",
        (1, 1, 1): base_path_sphere_high + "results_exp06_sphere_high_dim_new_version/curvature_errors_stats.json",
        (1, 100, 1): base_path_sphere_high + "results_exp07_sphere_high_dim_new_version/curvature_errors_stats.json",
        (0, 1, 1): base_path_sphere_high + "results_exp08_sphere_high_dim_new_version/curvature_errors_stats.json",
        (1, 1, 2): base_path_sphere_high + "results_exp09_sphere_high_dim_new_version/curvature_errors_stats.json",
        (1, 100, 2): base_path_sphere_high + "results_exp10_sphere_high_dim_new_version/curvature_errors_stats.json",
        (0, 1, 2): base_path_sphere_high + "results_exp11_sphere_high_dim_new_version/curvature_errors_stats.json",  
    },
    "Torus low dim": {
        (1, 0, 0): base_path_torus_low + "results_exp02_ae_t2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 1, 0): base_path_torus_low + "results_exp03_ae_t2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 100, 0): base_path_torus_low + "results_exp04_ae_t2_synthetic_w100_b64/curvature_errors_stats.json",
        (0, 1, 0): base_path_torus_low + "results_exp05_ae_t2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 1, 1): base_path_torus_low + "results_exp06_ae_t2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 100, 1): base_path_torus_low + "results_exp07_ae_t2_synthetic_w100_b64/curvature_errors_stats.json",
        (0, 1, 1): base_path_torus_low + "results_exp08_ae_t2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 1, 2): base_path_torus_low + "results_exp09_ae_t2_synthetic_w100_b64/curvature_errors_stats.json",
        (1, 100, 2): base_path_torus_low + "results_exp10_ae_t2_synthetic_w100_b64/curvature_errors_stats.json",
        (0, 1, 2): base_path_torus_low + "results_exp11_ae_t2_synthetic_w100_b64/curvature_errors_stats.json",  
    },
    "Torus high dim": {
        (1, 0, 0): base_path_torus_high + "results_exp02_torus_high_dim/curvature_errors_stats.json",
        (1, 1, 0): base_path_torus_high + "results_exp03_torus_high_dim/curvature_errors_stats.json",
        (1, 100, 0): base_path_torus_high + "results_exp04_torus_high_dim/curvature_errors_stats.json",
        (0, 1, 0): base_path_torus_high + "results_exp05_torus_high_dim/curvature_errors_stats.json",
        (1, 1, 1): base_path_torus_high + "results_exp06_torus_high_dim/curvature_errors_stats.json",
        (1, 100, 1): base_path_torus_high + "results_exp07_torus_high_dim/curvature_errors_stats.json",
        (0, 1, 1): base_path_torus_high + "results_exp08_torus_high_dim/curvature_errors_stats.json",
        (1, 1, 2): base_path_torus_high + "results_exp09_torus_high_dim/curvature_errors_stats.json",
        (1, 100, 2): base_path_torus_high + "results_exp10_torus_high_dim/curvature_errors_stats.json",
        (0, 1, 2): base_path_torus_high + "results_exp11_torus_high_dim/curvature_errors_stats.json",  
    },
}

datasets = [
    "Curve low dim", "Curve high dim",
    "Sphere low dim", "Sphere high dim",
    "Torus low dim", "Torus high dim"
]

generate_curv_table_from_json_lookup_alpha_empirical(
    file_lookup=file_lookup,
    datasets=datasets,
    save_path="tables/euclid_ae_full_table.tex"
)
generate_curv_table_from_json_lookup_alpha_empirical_norm(
    file_lookup=file_lookup,
    datasets=datasets,
    save_path="tables/euclid_ae_full_table_norm.tex"
)

LaTeX table written to: tables/euclid_ae_full_table.tex
LaTeX table written to: tables/euclid_ae_full_table_norm.tex


# Improved Normalization

In [15]:
import torch
import torch.nn.functional as F
from pathlib import Path

# ---------- Metric Computation ----------

def compute_bounding_box_diameter(x: torch.Tensor) -> float:
    min_vals, _ = x.min(dim=0)
    max_vals, _ = x.max(dim=0)
    return torch.norm(max_vals - min_vals, p=2).item()

def compute_curvature_metrics(pt_path: str):
    data = torch.load(pt_path, map_location='cpu', weights_only=False)
    inputs = data["points_sub"][0].float()
    latents = data["points_sub"][1].float()
    curv_inputs = torch.from_numpy(data["curvatures_sub"][1]).float()
    curv_latents = torch.from_numpy(data["curvatures_sub"][3]).float()

    assert curv_inputs.shape == curv_latents.shape

    diam_input = compute_bounding_box_diameter(inputs)
    diam_latent = compute_bounding_box_diameter(latents)
    scale = diam_input / diam_latent
    curv_latents_rescaled = curv_latents * (scale ** 2)

    # Unnormalized
    mse_raw = F.mse_loss(curv_latents, curv_inputs).item()
    smape_raw = 100 * torch.mean(torch.abs(curv_latents - curv_inputs) /
                                 (torch.abs(curv_latents) + torch.abs(curv_inputs)).clamp(min=1e-8)).item()

    # Normalized
    mse_scaled = F.mse_loss(curv_latents_rescaled, curv_inputs).item()
    smape_scaled = 100 * torch.mean(torch.abs(curv_latents_rescaled - curv_inputs) /
                                    (torch.abs(curv_latents_rescaled) + torch.abs(curv_inputs)).clamp(min=1e-8)).item()

    return [mse_raw, mse_scaled], [smape_raw, smape_scaled]


In [16]:
import json
import numpy as np

def _generate_curv_table_empirical(file_lookup, datasets, save_path, index, label):
    def format_value(val, unit="", highlight=None):
        if val == "-":
            return "-"
        v = f"{val:.2f}\\%" if unit == "%" else f"{val:.4f}"
        if highlight == "bold":
            return r"\textbf{" + v + "}"
        elif highlight == "underline":
            return r"\underline{" + v + "}"
        return v

    all_rows = []

    # Desired custom order
    param_order = [
        (1, 0, 0),
        (1, 1, 0),
        (1, 100, 0),
        (0, 1, 0),
        (1, 1, 1),
        (1, 100, 1),
        (0, 1, 1),
        (1, 1, 2),
        (1, 100, 2),
        (0, 1, 2),
    ]
    
    for alpha, gamma, d in param_order:
        ds_keys = [d]
    
        for d_curr in ds_keys:
            meta = (f"{alpha}", f"{gamma}", f"{d_curr}")
            mse_vals, smape_vals = [], []
    
            for ds in datasets:
                path = file_lookup.get(ds, {}).get((alpha, gamma, d_curr), None)
                if not path:
                    mse_vals.append("-")
                    smape_vals.append("-")
                    continue
                try:
                    mse, smape = compute_curvature_metrics(path)
                    mse_vals.append(mse[index])
                    smape_vals.append(smape[index])
                except Exception as e:
                    print(f"Error reading {path}: {e}")
                    mse_vals.append("-")
                    smape_vals.append("-")
    
            all_rows.append((meta, mse_vals, smape_vals))


    # Highlight minima
    n_datasets = len(datasets)
    mse_col, smape_col = [[] for _ in range(n_datasets)], [[] for _ in range(n_datasets)]
    for rid, (_, mse_vals, smape_vals) in enumerate(all_rows):
        for i in range(n_datasets):
            if mse_vals[i] != "-":
                mse_col[i].append((rid, mse_vals[i]))
            if smape_vals[i] != "-":
                smape_col[i].append((rid, smape_vals[i]))

    mse_highlight, smape_highlight = {}, {}
    for i in range(n_datasets):
        if mse_col[i]:
            sorted_m = sorted(mse_col[i], key=lambda x: x[1])
            mse_highlight[(sorted_m[0][0], i)] = "bold"
            if len(sorted_m) > 1:
                mse_highlight[(sorted_m[1][0], i)] = "underline"
        if smape_col[i]:
            sorted_s = sorted(smape_col[i], key=lambda x: x[1])
            smape_highlight[(sorted_s[0][0], i)] = "bold"
            if len(sorted_s) > 1:
                smape_highlight[(sorted_s[1][0], i)] = "underline"

    # Render LaTeX
    lines = []
    lines.append(r"\begin{table}[ht]")
    lines.append(r"\scriptsize")
    lines.append(r"    \centering")
    lines.append(f"    \\begin{{tabular}}{{lll|{'cc|' * n_datasets}}}")
    lines.append(r"        \toprule")
    lines.append("        $\\alpha$ & $\\gamma$ & $d_{\\text{topo}}$" +
                 "".join([f" & \\multicolumn{{2}}{{c|}}{{{ds}}}" for ds in datasets]) + r" \\")
    lines.append("        & & & " + " & ".join(["MSE & SMAPE"] * n_datasets) + r" \\")
    lines.append(r"        \midrule")

    for rid, (meta, mse_vals, smape_vals) in enumerate(all_rows):
        alpha, gamma, dtopo = meta
        dtopo_display = "-" if gamma == "0" else dtopo
        row = [alpha, gamma, dtopo_display]
        for i in range(n_datasets):
            mse_fmt = format_value(mse_vals[i], highlight=mse_highlight.get((rid, i)))
            smape_fmt = format_value(smape_vals[i], unit="%", highlight=smape_highlight.get((rid, i)))
            row.extend([mse_fmt, smape_fmt])
        lines.append("        " + " & ".join(row) + r" \\")

    lines.append(r"        \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(rf"    \caption{{Curvature metrics from index {index} across models with varying $\alpha$, $\gamma$, $d_{{\text{{topo}}}}$.}}")
    lines.append(rf"    \label{{tab:{label}}}")
    lines.append(r"\end{table}")

    with open(save_path, "w") as f:
        f.write("\n".join(lines))
    print(f"LaTeX table written to: {save_path}")

def generate_curv_table_from_json_lookup_alpha_empirical(file_lookup, datasets, save_path):
    _generate_curv_table_empirical(file_lookup, datasets, save_path, index=0, label="curv_alpha_empirical")

def generate_curv_table_from_json_lookup_alpha_empirical_norm(file_lookup, datasets, save_path):
    _generate_curv_table_empirical(file_lookup, datasets, save_path, index=1, label="curv_alpha_empirical_norm")


In [17]:
base_path = "notebooks_euclid_ae/curvatures/"

file_lookup = {
    "Curve low dim": {
        (1, 0, 0): base_path + "curvatures_exp01_ae_flower_curve_w100_b64.pt",
        (1, 1, 0): base_path + "curvatures_exp02_ae_flower_curve_w100_b64.pt",
        (1, 100, 0): base_path + "curvatures_exp03_ae_flower_curve_w100_b64.pt",
        (0, 1, 0): base_path + "curvatures_exp04_ae_flower_curve_w100_b64.pt",
        (1, 1, 1): base_path + "curvatures_exp05_ae_flower_curve_w100_b64.pt",
        (1, 100, 1): base_path + "curvatures_exp06_ae_flower_curve_w100_b64.pt",
        (0, 1, 1): base_path + "curvatures_exp07_ae_flower_curve_w100_b64.pt",
        (1, 1, 2): base_path + "curvatures_exp08_ae_flower_curve_w100_b64.pt",
        (1, 100, 2): base_path + "curvatures_exp09_ae_flower_curve_w100_b64.pt",
        (0, 1, 2): base_path + "curvatures_exp10_ae_flower_curve_w100_b64.pt",
    },
    "Curve high dim": {
        (1, 0, 0): base_path + "curvatures_exp01_ae_scrunchy_w100_b64.pt",
        (1, 1, 0): base_path + "curvatures_exp02_ae_scrunchy_w100_b64.pt",
        (1, 100, 0): base_path + "curvatures_exp03_ae_scrunchy_w100_b64.pt",
        (0, 1, 0): base_path + "curvatures_exp04_ae_scrunchy_w100_b64.pt",
        (1, 1, 1): base_path + "curvatures_exp05_ae_scrunchy_w100_b64.pt",
        (1, 100, 1): base_path + "curvatures_exp06_ae_scrunchy_w100_b64.pt",
        (0, 1, 1): base_path + "curvatures_exp07_ae_scrunchy_w100_b64.pt",
        (1, 1, 2): base_path + "curvatures_exp08_ae_scrunchy_w100_b64.pt",
        (1, 100, 2): base_path + "curvatures_exp09_ae_scrunchy_w100_b64.pt",
        (0, 1, 2): base_path + "curvatures_exp10_ae_scrunchy_w100_b64.pt",
    },
    "Sphere low dim": {
        (1, 0, 0): base_path + "curvatures_exp02_ae_s2_synthetic_w100_b64.pt",
        (1, 1, 0): base_path + "curvatures_exp03_ae_s2_synthetic_w100_b64.pt",
        (1, 100, 0): base_path + "curvatures_exp04_ae_s2_synthetic_w100_b64.pt",
        (0, 1, 0): base_path + "curvatures_exp05_ae_s2_synthetic_w100_b64.pt",
        (1, 1, 1): base_path + "curvatures_exp06_ae_s2_synthetic_w100_b64.pt",
        (1, 100, 1): base_path + "curvatures_exp07_ae_s2_synthetic_w100_b64.pt",
        (0, 1, 1): base_path + "curvatures_exp08_ae_s2_synthetic_w100_b64.pt",
        (1, 1, 2): base_path + "curvatures_exp09_ae_s2_synthetic_w100_b64.pt",
        (1, 100, 2): base_path + "curvatures_exp10_ae_s2_synthetic_w100_b64.pt",
        (0, 1, 2): base_path + "curvatures_exp11_ae_s2_synthetic_w100_b64.pt",
    },
    "Sphere high dim": {
        (1, 0, 0): base_path + "curvatures_exp02_sphere_high_dim_new_version.pt",
        (1, 1, 0): base_path + "curvatures_exp03_sphere_high_dim_new_version.pt",
        (1, 100, 0): base_path + "curvatures_exp04_sphere_high_dim_new_version.pt",
        (0, 1, 0): base_path + "curvatures_exp05_sphere_high_dim_new_version.pt",
        (1, 1, 1): base_path + "curvatures_exp06_sphere_high_dim_new_version.pt",
        (1, 100, 1): base_path + "curvatures_exp07_sphere_high_dim_new_version.pt",
        (0, 1, 1): base_path + "curvatures_exp08_sphere_high_dim_new_version.pt",
        (1, 1, 2): base_path + "curvatures_exp09_sphere_high_dim_new_version.pt",
        (1, 100, 2): base_path + "curvatures_exp10_sphere_high_dim_new_version.pt",
        (0, 1, 2): base_path + "curvatures_exp11_sphere_high_dim_new_version.pt",  
    },
    "Torus low dim": {
        (1, 0, 0): base_path + "curvatures_exp02_ae_t2_synthetic_w100_b64.pt",
        (1, 1, 0): base_path + "curvatures_exp03_ae_t2_synthetic_w100_b64.pt",
        (1, 100, 0): base_path + "curvatures_exp04_ae_t2_synthetic_w100_b64.pt",
        (0, 1, 0): base_path + "curvatures_exp05_ae_t2_synthetic_w100_b64.pt",
        (1, 1, 1): base_path + "curvatures_exp06_ae_t2_synthetic_w100_b64.pt",
        (1, 100, 1): base_path + "curvatures_exp07_ae_t2_synthetic_w100_b64.pt",
        (0, 1, 1): base_path + "curvatures_exp08_ae_t2_synthetic_w100_b64.pt",
        (1, 1, 2): base_path + "curvatures_exp09_ae_t2_synthetic_w100_b64.pt",
        (1, 100, 2): base_path + "curvatures_exp10_ae_t2_synthetic_w100_b64.pt",
        (0, 1, 2): base_path + "curvatures_exp11_ae_t2_synthetic_w100_b64.pt",  
    },
    "Torus high dim": {
        (1, 0, 0): base_path + "curvatures_exp02_torus_high_dim.pt",
        (1, 1, 0): base_path + "curvatures_exp03_torus_high_dim.pt",
        (1, 100, 0): base_path + "curvatures_exp04_torus_high_dim.pt",
        (0, 1, 0): base_path + "curvatures_exp05_torus_high_dim.pt",
        (1, 1, 1): base_path + "curvatures_exp06_torus_high_dim.pt",
        (1, 100, 1): base_path + "curvatures_exp07_torus_high_dim.pt",
        (0, 1, 1): base_path + "curvatures_exp08_torus_high_dim.pt",
        (1, 1, 2): base_path + "curvatures_exp09_torus_high_dim.pt",
        (1, 100, 2): base_path + "curvatures_exp10_torus_high_dim.pt",
        (0, 1, 2): base_path + "curvatures_exp11_torus_high_dim.pt",  
    },
}


datasets = [
    "Curve low dim", "Curve high dim",
    "Sphere low dim", "Sphere high dim",
    "Torus low dim", "Torus high dim"
]

generate_curv_table_from_json_lookup_alpha_empirical(
    file_lookup=file_lookup,
    datasets=datasets,
    save_path="tables/euclid_ae_full_table.tex"
)
generate_curv_table_from_json_lookup_alpha_empirical_norm(
    file_lookup=file_lookup,
    datasets=datasets,
    save_path="tables/euclid_ae_full_table_norm.tex"
)

LaTeX table written to: tables/euclid_ae_full_table.tex
LaTeX table written to: tables/euclid_ae_full_table_norm.tex
